In [28]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest, f_classif, chi2

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from lightgbm import LGBMClassifier 
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid, RadiusNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF


# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval
import importlib
importlib.reload(model_eval)

# Class Imbalance
from imblearn.over_sampling import SMOTE, RandomOverSampler

## Imports

In [3]:
X_train= pd.read_csv(
    filepath_or_buffer= "../data/processed/X_train_probability.csv"
)
X_train.info()
X_train.head(1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22686 entries, 0 to 22685
Data columns (total 32 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numeric__person_age_1.0                      22686 non-null  float64
 1   numeric__person_age_2.0                      22686 non-null  float64
 2   numeric__person_income_1.0                   22686 non-null  float64
 3   numeric__person_income_2.0                   22686 non-null  float64
 4   numeric__person_emp_length_1.0               22686 non-null  float64
 5   numeric__person_emp_length_2.0               22686 non-null  float64
 6   numeric__loan_amnt_1.0                       22686 non-null  float64
 7   numeric__loan_amnt_2.0                       22686 non-null  float64
 8   numeric__loan_int_rate_1.0                   22686 non-null  float64
 9   numeric__loan_int_rate_2.0                   22686 non-null  float64
 10

,numeric__person_age_1.0,numeric__person_age_2.0,numeric__person_income_1.0,numeric__person_income_2.0,numeric__person_emp_length_1.0,numeric__person_emp_length_2.0,numeric__loan_amnt_1.0,numeric__loan_amnt_2.0,numeric__loan_int_rate_1.0,numeric__loan_int_rate_2.0,...,categorical__loan_intent_PERSONAL,categorical__loan_intent_VENTURE,categorical__loan_grade_A,categorical__loan_grade_B,categorical__loan_grade_C,categorical__loan_grade_D,categorical__loan_grade_E,categorical__loan_grade_F,categorical__loan_grade_G,categorical__cb_person_default_on_file_True
0,-0.728894,1.261899,-0.698358,1.394801,-0.669061,-0.747494,-0.729957,1.411273,1.396722,-0.707107,...,-0.452031,-0.461055,-0.701014,1.460921,-0.500744,-0.354762,-0.174851,-0.086634,-0.045564,-0.465617


In [4]:
X_test= pd.read_csv(
    filepath_or_buffer= "../data/processed/X_test_probability.csv"
)
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9723 entries, 0 to 9722
Data columns (total 32 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numeric__person_age_1.0                      9723 non-null   float64
 1   numeric__person_age_2.0                      9723 non-null   float64
 2   numeric__person_income_1.0                   9723 non-null   float64
 3   numeric__person_income_2.0                   9723 non-null   float64
 4   numeric__person_emp_length_1.0               9723 non-null   float64
 5   numeric__person_emp_length_2.0               9723 non-null   float64
 6   numeric__loan_amnt_1.0                       9723 non-null   float64
 7   numeric__loan_amnt_2.0                       9723 non-null   float64
 8   numeric__loan_int_rate_1.0                   9723 non-null   float64
 9   numeric__loan_int_rate_2.0                   9723 non-null   float64
 10  

In [5]:
y_train= pd.read_csv(
    filepath_or_buffer= "../data/interim/y_train.csv"
)
y_train = y_train.values.ravel()

In [6]:
y_test= pd.read_csv(
    filepath_or_buffer= "../data/interim/y_test.csv"
)
y_test = y_test.values.ravel()

## Comparing Models

In [16]:
untuned_models = [
    BernoulliNB(),
    LogisticRegression(
        C=np.inf,
        random_state=42
    ),
    SGDClassifier(
        loss= 'log_loss',
        penalty=None,
        random_state=42,
        n_jobs=-1
    ),
    LinearDiscriminantAnalysis(), #  handles internal feature selection with n_components
    # GaussianProcessClassifier(
    #     kernel=kernel,
    #     random_state=42,
    #     n_jobs=-1
    # )
]

In [17]:
untuned_model_performance, untuned_fitted_models = model_eval.comparing_models(
    untuned_models,
    X_train,
    y_train,
    X_test,
    y_test
)

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\linear_model\_logistic.py:1170: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


In [19]:
untuned_model_performance

,model,roc_auc,pr_auc,log_loss,brier_score,matthews_corrcoef
1,"LogisticRegression (penalty=deprecated, solver...",0.854396,0.673815,0.365527,0.113222,0.480977
3,LinearDiscriminantAnalysis (solver=svd),0.851276,0.661966,0.376388,0.116366,0.477959
2,"SGDClassifier (penalty=None, n_jobs=-1, l1_rat...",0.838913,0.653276,0.382267,0.117396,0.476628
0,BernoulliNB,0.828282,0.618647,0.451205,0.137161,0.435658


## Hyperparameter Tuning, Class Imbalance, Feature Selection

### Logistic Regression (L1, L2, ElasticNet)

Hyperparameter tuning the Cs regularization strength for three differet regularization methods. Class Imbalance is handled internally. Feature Selection is handled internally via regularization.

In [39]:
tuned_models=[
        LogisticRegressionCV(Cs= np.logspace(-3, 3, 10), solver='saga', max_iter=5000, class_weight='balanced', n_jobs=-1, l1_ratios=(1,), use_legacy_attributes=False),
        LogisticRegressionCV(Cs= np.logspace(-3, 3, 10), solver='saga', max_iter=5000, class_weight='balanced', n_jobs=-1, l1_ratios=(0,), use_legacy_attributes=False),
        LogisticRegressionCV(Cs= np.logspace(-3, 3, 10), solver='saga', max_iter=5000, class_weight='balanced', n_jobs=-1, l1_ratios=[0.5], use_legacy_attributes=False),
]

In [40]:
internal_tuned_model_performance, fitted_models_internal = model_eval.comparing_models(
    tuned_models,
    X_train,
    y_train,
    X_test,
    y_test
)

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Linear Discriminant Analysis

In [41]:
lda_pipe = Pipeline(
    steps=[
        ("select", SelectKBest(score_func=f_classif)),
        ("ros", RandomOverSampler()),
        ("lda", LinearDiscriminantAnalysis())
    ]
)

In [42]:
lda_param_dist = {
    "select__k": stats.randint(5, 300),
    "lda__solver": ["svd, lsqr", "eigen"],
    "lda__shrinkage": ["auto", stats.uniform(0, 1)]
}

In [43]:
lda_tuned = HalvingRandomSearchCV(
    estimator= lda_pipe,
    param_distributions=lda_param_dist,
    scoring = 'roc_auc',
    n_jobs=-1
)

### SGDClassifier (L1)

In [44]:
sgd_classifier = SGDClassifier(
    loss= 'log_loss',
    max_iter= 10000,
    penalty= 'l1',
    class_weight= 'balanced',
    n_jobs=-1
)

In [45]:
sgd_param_grid = {
    'alpha': loguniform(1e-2, 1e6),
}

In [46]:
sgd_tuned = HalvingRandomSearchCV(
    estimator= sgd_classifier,
    param_distributions=sgd_param_grid,
    scoring = 'roc_auc',
    n_jobs=-1
)

### BernoulliNB

In [47]:
nb_pipe = Pipeline(
    steps=[
        ('select', SelectKBest(score_func=chi2)),
        ('sampling', RandomOverSampler()),
        ('nb', BernoulliNB())
    ]
)

In [48]:
nb_param_grid = {
    'nb__alpha': stats.loguniform(1e-4, 1e1)
}

In [49]:
nb_tuned = HalvingRandomSearchCV(
    estimator= nb_pipe,
    param_distributions=nb_param_grid,
    scoring = 'roc_auc',
    n_jobs=-1
)

In [57]:
manually_tuned_models = [
    lda_tuned,
    nb_tuned,
    sgd_tuned
]

In [58]:
lda_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('select', ...), ('ros', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...0020A42C945E0>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",10
,sampling_strategy,'auto'
,random_state,None
,shrinkage,None
,"solver solver: {'svd', 'lsqr', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'lsqr': Least squares solution. Can be combined with shrinkage or custom covariance estimator. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator... versionchanged:: 1.2 `solver=""svd""` now has experimental Array API support. See the :ref:`Array API User Guide ` for more details.",'svd'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter.This should be left to None if `covariance_estimator` is used.Note that shrinkage works only with 'lsqr' and 'eigen' solvers.For a usage example, see:ref:`sphx_glr_auto_examples_classification_plot_lda.py`.",None


In [59]:
manually_tuned_model_performance, fitted_models_manual= model_eval.comparing_manually_tuned_models(
    manually_tuned_models,
    X_train,
    y_train,
    X_test,
    y_test
)

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan ... nan nan nan]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.5        0.78243386 0.5       ]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.75197294 0.76646027 0.77500549]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.7764742  0.72429461 0.73309415]
  warnings.warn(
c:\Use

In [60]:
tuned_model_performances = model_eval.combine_model_performance(
    internal_tuned_model_performance,
    manually_tuned_model_performance
)

### Evaluate Calibration